# Stage 1a — YOLO Training (Kaggle, 2× T4)

Fine-tunes YOLO11x on the surveillance composite dataset, with fixes for the three problems found in the previous run:

| Problem in previous run | Fix applied here |
|---|---|
| `Vehicle` class had **0 instances** (`nc: 7`, only 6 reported) | Merged into `Car`, `nc: 6` |
| `Carrying_object` mAP50 = **0.088**, recall **0.066** | Oversampling + `copy_paste` augmentation |
| `Person` recall **0.604** (40% missed) | Recall-oriented confidence sweep after training |

**Previous baseline to beat:** mAP50 `0.663` / mAP50-95 `0.461`, Person recall `0.604`, Carrying_object mAP50 `0.088`.

**Gate to pass before Stage 1b (ByteTrack):** Person recall ≥ 0.80, Carrying_object mAP50 ≥ 0.45.

> Settings: `Accelerator → GPU T4 ×2`, `Internet → On`.

## 0. Setup

In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 6.5 MB/s eta 0:00:00


In [2]:
import os, shutil, random, glob
from collections import Counter, defaultdict
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

SEED = 42
random.seed(SEED)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Torch: 2.10.0+cu128 | CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## 1. Configuration

Every threshold lives here — nothing hardcoded further down.

In [3]:
# ---- Paths ----
# ---- Paths ----
SOURCE = "/kaggle/input/datasets/chaitanyazunzurkar/surveillance-dataset/surveillance_composite"
WORK   = "/kaggle/working/surveillance_v2"
RUNS   = "/kaggle/working/runs"

# ---- Original class map (7 classes, as previously trained) ----
OLD_NAMES = {0:"Person", 1:"Car", 2:"Vehicle", 3:"Carrying_object",
             4:"Bike", 5:"Gun", 6:"Knife"}

# ---- New class map: Vehicle merged into Car, renumbered to 6 classes ----
NEW_NAMES = {0:"Person", 1:"Car", 2:"Carrying_object", 3:"Bike", 4:"Gun", 5:"Knife"}

# old_id -> new_id  (2 "Vehicle" folds into 1 "Car")
CLASS_REMAP = {0:0, 1:1, 2:1, 3:2, 4:3, 5:4, 6:5}

RARE_CLASSES = {2}          # new-id of Carrying_object
OVERSAMPLE_FACTOR = 6       # duplicate rare-class train images this many times

VAL_FRACTION = 0.20

# ---- Training ----
EPOCHS   = 100
IMGSZ    = 640
BATCH    = 16
MODEL    = "yolo11x.pt"
RUN_NAME = "yolo11x_surv_v1"

print("New classes:", NEW_NAMES)

New classes: {0: 'Person', 1: 'Car', 2: 'Carrying_object', 3: 'Bike', 4: 'Gun', 5: 'Knife'}


## 2. Diagnose the source label distribution

Before changing anything, confirm the imbalance that broke `Carrying_object` — and verify `Vehicle` really is empty.

In [4]:
SRC_IMG = f"{SOURCE}/images/train"
SRC_LBL = f"{SOURCE}/labels/train"

all_images = sorted([f for f in os.listdir(SRC_IMG)
                     if f.lower().endswith((".jpg", ".jpeg", ".png"))])
print("Total images:", len(all_images))

def read_label(path):
    """Return list of (class_id, rest_of_line) for a YOLO label file."""
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 5:
                rows.append((int(parts[0]), " ".join(parts[1:])))
    return rows

inst_counter = Counter()      # instances per old class
img_counter  = Counter()      # images containing each old class
missing_labels = 0

for img in all_images:
    lbl = os.path.join(SRC_LBL, os.path.splitext(img)[0] + ".txt")
    rows = read_label(lbl)
    if not os.path.exists(lbl):
        missing_labels += 1
        continue
    present = set()
    for cid, _ in rows:
        inst_counter[cid] += 1
        present.add(cid)
    for cid in present:
        img_counter[cid] += 1

print(f"Images without a label file: {missing_labels}\n")
print(f"{'old_id':>6}  {'name':<18}{'instances':>10}{'images':>9}   share")
total_inst = sum(inst_counter.values())
for cid in sorted(OLD_NAMES):
    n = inst_counter.get(cid, 0)
    share = (n / total_inst * 100) if total_inst else 0
    flag = "   <-- EMPTY" if n == 0 else ("   <-- RARE" if share < 2 else "")
    print(f"{cid:>6}  {OLD_NAMES[cid]:<18}{n:>10}{img_counter.get(cid,0):>9}   {share:5.2f}%{flag}")

Total images: 6874
Images without a label file: 145

old_id  name               instances   images   share
     0  Person                 38878     4431   51.69%
     1  Car                    30507     1807   40.56%
     2  Vehicle                    0        0    0.00%   <-- EMPTY
     3  Carrying_object          392      225    0.52%   <-- RARE
     4  Bike                    1129      274    1.50%   <-- RARE
     5  Gun                     2068     1661    2.75%
     6  Knife                   2245     1937    2.98%


## 3. Rebuild the dataset: remap classes, split, oversample rare class

Three things happen here, in order:

1. **Remap** every label file to the new 6-class scheme (`Vehicle` → `Car`).
2. **Split** 80/20 into train/val — images with *no* label file are dropped entirely.
3. **Oversample** train images containing `Carrying_object`, so the rare class stops being ~0.5% of the data.

Oversampling is applied to **train only**. The val split stays untouched and honest — otherwise the metric would be inflated by duplicates.

In [5]:
# Keep only images that actually have a label file
paired = []
for img in all_images:
    lbl = os.path.join(SRC_LBL, os.path.splitext(img)[0] + ".txt")
    if os.path.exists(lbl):
        paired.append(img)

print(f"Usable image/label pairs: {len(paired)} (dropped {len(all_images)-len(paired)})")

random.shuffle(paired)
n_val = int(len(paired) * VAL_FRACTION)
val_images   = paired[:n_val]
train_images = paired[n_val:]
print(f"Train: {len(train_images)} | Val: {len(val_images)}")

Usable image/label pairs: 6729 (dropped 145)
Train: 5384 | Val: 1345


In [6]:
# Fresh working dirs
if os.path.exists(WORK):
    shutil.rmtree(WORK)
for d in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)


def write_remapped_label(src_lbl, dst_lbl):
    """Remap old class ids -> new ids. Returns set of new ids written."""
    out, present = [], set()
    for cid, rest in read_label(src_lbl):
        if cid not in CLASS_REMAP:
            continue
        new_id = CLASS_REMAP[cid]
        out.append(f"{new_id} {rest}")
        present.add(new_id)
    with open(dst_lbl, "w") as f:
        f.write("\n".join(out) + ("\n" if out else ""))
    return present


def materialise(image_list, split, oversample=False):
    """Copy images+labels into WORK. Duplicates rare-class images when oversample=True."""
    n_written, n_dupes = 0, 0
    for img in image_list:
        stem = os.path.splitext(img)[0]
        src_i = f"{SRC_IMG}/{img}"
        src_l = f"{SRC_LBL}/{stem}.txt"

        dst_i = f"{WORK}/images/{split}/{img}"
        dst_l = f"{WORK}/labels/{split}/{stem}.txt"
        shutil.copy2(src_i, dst_i)
        present = write_remapped_label(src_l, dst_l)
        n_written += 1

        if oversample and (present & RARE_CLASSES):
            for k in range(1, OVERSAMPLE_FACTOR):
                dup = f"{stem}__os{k}"
                shutil.copy2(src_i, f"{WORK}/images/{split}/{dup}{os.path.splitext(img)[1]}")
                write_remapped_label(src_l, f"{WORK}/labels/{split}/{dup}.txt")
                n_dupes += 1
    return n_written, n_dupes


tr_n, tr_dup = materialise(train_images, "train", oversample=True)
va_n, va_dup = materialise(val_images,   "val",   oversample=False)

print(f"Train: {tr_n} originals + {tr_dup} oversampled duplicates = {tr_n + tr_dup}")
print(f"Val:   {va_n} originals (no oversampling — kept honest)")

Train: 5384 originals + 920 oversampled duplicates = 6304
Val:   1345 originals (no oversampling — kept honest)


In [7]:
# Verify the new balance
def count_split(split):
    inst, imgs = Counter(), Counter()
    for lbl in glob.glob(f"{WORK}/labels/{split}/*.txt"):
        present = set()
        for cid, _ in read_label(lbl):
            inst[cid] += 1
            present.add(cid)
        for cid in present:
            imgs[cid] += 1
    return inst, imgs

for split in ["train", "val"]:
    inst, imgs = count_split(split)
    total = sum(inst.values())
    print(f"\n--- {split.upper()} ---")
    print(f"{'id':>3}  {'name':<18}{'instances':>10}{'images':>9}   share")
    for cid in sorted(NEW_NAMES):
        n = inst.get(cid, 0)
        share = (n / total * 100) if total else 0
        print(f"{cid:>3}  {NEW_NAMES[cid]:<18}{n:>10}{imgs.get(cid,0):>9}   {share:5.2f}%")


--- TRAIN ---
 id  name               instances   images   share
  0  Person                 36005     4454   47.42%
  1  Car                    33350     1964   43.92%
  2  Carrying_object         1992     1104    2.62%
  3  Bike                    1179      319    1.55%
  4  Gun                     1639     1323    2.16%
  5  Knife                   1767     1532    2.33%

--- VAL ---
 id  name               instances   images   share
  0  Person                  8093      877   56.19%
  1  Car                     5157      343   35.81%
  2  Carrying_object           60       41    0.42%
  3  Bike                     185       45    1.28%
  4  Gun                      429      338    2.98%
  5  Knife                    478      405    3.32%


### Checkpoint

`Carrying_object` share in **train** should now be meaningfully higher than the ~0.5% it was.
If it is still under ~3%, raise `OVERSAMPLE_FACTOR` and re-run from section 3 — or better, source more carried-object images, since duplicates add no new visual variety.

## 4. Write the dataset YAML

In [8]:
DATA_YAML = f"{WORK}/surveillance_v2.yaml"

data_cfg = {
    "path":  WORK,
    "train": "images/train",
    "val":   "images/val",
    "names": {int(k): v for k, v in NEW_NAMES.items()},
    "nc":    len(NEW_NAMES),
}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print(open(DATA_YAML).read())

path: /kaggle/working/surveillance_v2
train: images/train
val: images/val
names:
  0: Person
  1: Car
  2: Carrying_object
  3: Bike
  4: Gun
  5: Knife
nc: 6



## 5. Train

Augmentation notes — these two are the rare-class levers:

- **`copy_paste=0.3`** pastes object instances between images. This is the single most effective
  augmentation for rare-class instance counts, because it creates genuinely new
  `Carrying_object` placements rather than duplicate frames.
- **`mosaic=1.0`** (default) tiles four images together, which also raises the chance a rare
  instance appears in any given batch.

`scale` and `fliplr` are kept moderate; surveillance cameras are fixed, so aggressive geometric
distortion teaches viewpoints the model will never see.

In [9]:
model = YOLO(MODEL)

results = model.train(
    data=DATA_YAML,

    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=[0, 1],
    workers=4,
    pretrained=True,
    seed=SEED,

    # --- rare-class augmentation ---
    copy_paste=0.3,
    mosaic=1.0,
    close_mosaic=10,
    scale=0.5,
    fliplr=0.5,
    degrees=0.0,       # fixed cameras: no rotation

    # --- checkpointing ---
    project=RUNS,
    name=RUN_NAME,
    save=True,
    save_period=10,
    patience=30,       # early-stop if no improvement
    plots=True,
)

Ultralytics 8.4.158 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/surveillance_v2/surveillance_v2.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mode

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      8.52G      1.654      1.366      1.391        197        640: 100% ━━━━━━━━━━━━ 379/379 1.8it/s 3:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 3.4it/s 11.9s
                   all       1283      13449       0.47      0.408      0.388      0.207

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      8.56G      1.642      1.381      1.397        117        640: 100% ━━━━━━━━━━━━ 379/379 1.8it/s 3:29
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 3.4it/s 12.2s
                   all       1283      13449      0.446      0.378       0.36      0.168

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      8.29G      1.583      1.295      1.358         71        640: 100% ━━━━━━━━━

## 6. Per-class evaluation

The aggregate mAP hides the problem — always read the per-class table.

In [10]:
BEST = f"{RUNS}/{RUN_NAME}/weights/best.pt"
best_model = YOLO(BEST)

metrics = best_model.val(data=DATA_YAML, split="val", imgsz=IMGSZ)

PREV = {  # previous run, for comparison
    "Person":          dict(R=0.604, mAP50=0.680),
    "Car":             dict(R=0.803, mAP50=0.814),
    "Carrying_object": dict(R=0.066, mAP50=0.088),
    "Bike":            dict(R=0.478, mAP50=0.568),
    "Gun":             dict(R=0.925, mAP50=0.947),
    "Knife":           dict(R=0.857, mAP50=0.883),
}

print(f"\n{'class':<18}{'P':>7}{'R':>7}{'mAP50':>8}{'mAP50-95':>10}   {'prev mAP50':>11}  delta")
print("-" * 72)
for i, name in NEW_NAMES.items():
    try:
        p, r, ap50, ap = metrics.class_result(i)
    except Exception:
        continue
    prev = PREV.get(name, {}).get("mAP50")
    if prev is not None:
        d = ap50 - prev
        print(f"{name:<18}{p:>7.3f}{r:>7.3f}{ap50:>8.3f}{ap:>10.3f}   {prev:>11.3f}  {d:+.3f}")
    else:
        print(f"{name:<18}{p:>7.3f}{r:>7.3f}{ap50:>8.3f}{ap:>10.3f}   {'—':>11}")

print("-" * 72)
print(f"{'ALL':<18}{metrics.box.mp:>7.3f}{metrics.box.mr:>7.3f}"
      f"{metrics.box.map50:>8.3f}{metrics.box.map:>10.3f}   {0.663:>11.3f}  {metrics.box.map50-0.663:+.3f}")

Ultralytics 8.4.158 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11x summary (fused): 190 layers, 56,833,954 parameters, 0 gradients, 194.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1933.4±1292.3 MB/s, size: 341.0 KB)
val: Scanning /kaggle/working/surveillance_v2/labels/val.cache... 1345 images, 0 backgrounds, 62 corrupt: 100% ━━━━━━━━━━━━ 1345/1345 331.8Mit/s 0.0s
val: /kaggle/working/surveillance_v2/images/val/crowdhuman_273271,165ab00051493658.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1324      1.1574]
val: /kaggle/working/surveillance_v2/images/val/crowdhuman_273271,1f417000f6436bb9.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.4558      1.5553      1.4602]
val: /kaggle/working/surveillance_v2/images/val/crowdhuman_273271,20abf000ec164714.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0335]
val: /kaggle/working/surveilla

## 7. Gate check

Stage 1 is not finished until both conditions hold. Tracking a class detected 6% of the time
produces fragmented, unusable tracks — so this gate protects Stage 1b, not just the metric.

In [11]:
def class_index(name):
    return [k for k, v in NEW_NAMES.items() if v == name][0]

p_r    = metrics.class_result(class_index("Person"))[1]
co_ap  = metrics.class_result(class_index("Carrying_object"))[2]

gates = [
    ("Person recall          >= 0.80", p_r,   0.80),
    ("Carrying_object mAP50  >= 0.45", co_ap, 0.45),
]

print("GATE CHECK\n" + "=" * 46)
all_pass = True
for label, value, target in gates:
    ok = value >= target
    all_pass &= ok
    print(f"{'PASS' if ok else 'FAIL'}  {label}   (got {value:.3f})")
print("=" * 46)
print("\nProceed to ByteTrack." if all_pass else
      "\nDo NOT proceed yet. Raise OVERSAMPLE_FACTOR, add real "
      "Carrying_object images, or train longer, then re-run.")

GATE CHECK
FAIL  Person recall          >= 0.80   (got 0.608)
FAIL  Carrying_object mAP50  >= 0.45   (got 0.068)

Do NOT proceed yet. Raise OVERSAMPLE_FACTOR, add real Carrying_object images, or train longer, then re-run.


## 8. Confidence sweep — tune for recall, not F1

ByteTrack's second association pass is *designed* to consume low-confidence boxes, so the usual
instinct to maximise F1 is wrong here. A missed detection is unrecoverable; a spurious low-confidence
box is cheap, because tracking will simply fail to associate it and drop it.

Pick the lowest confidence where precision is still tolerable (roughly ≥ 0.55).

In [12]:
print(f"{'conf':>6}{'P':>9}{'R':>9}{'mAP50':>9}")
print("-" * 34)
sweep = {}
for conf in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.50]:
    m = best_model.val(data=DATA_YAML, split="val", imgsz=IMGSZ,
                       conf=conf, verbose=False, plots=False)
    sweep[conf] = (m.box.mp, m.box.mr, m.box.map50)
    print(f"{conf:>6.2f}{m.box.mp:>9.3f}{m.box.mr:>9.3f}{m.box.map50:>9.3f}")

print("\nPick the LOWEST conf with precision >= ~0.55, then set it in Stage 1b.")

  conf        P        R    mAP50
----------------------------------
Ultralytics 8.4.158 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11x summary (fused): 190 layers, 56,833,954 parameters, 0 gradients, 194.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2270.7±1283.1 MB/s, size: 648.7 KB)
val: Scanning /kaggle/working/surveillance_v2/labels/val.cache... 1345 images, 0 backgrounds, 62 corrupt: 100% ━━━━━━━━━━━━ 1345/1345 376.1Mit/s 0.0s
val: /kaggle/working/surveillance_v2/images/val/crowdhuman_273271,165ab00051493658.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1324      1.1574]
val: /kaggle/working/surveillance_v2/images/val/crowdhuman_273271,1f417000f6436bb9.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.4558      1.5553      1.4602]
val: /kaggle/working/surveillance_v2/images/val/crowdhuman_273271,20abf000ec164714.jpg: ignoring corrupt image/label: non-normalized or o

In [13]:
# Lock the chosen value here — Stage 1b (ByteTrack) imports this.
CONF_THRESHOLD = 0.20      # <-- set from the sweep above

cfg = {
    "weights": "yolo11x_surv_v1.pt",
    "conf_threshold": CONF_THRESHOLD,
    "imgsz": IMGSZ,
    "names": {int(k): v for k, v in NEW_NAMES.items()},
    "nc": len(NEW_NAMES),
    "val_map50": float(metrics.box.map50),
    "val_map50_95": float(metrics.box.map),
}

with open("/kaggle/working/stage1_config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(open("/kaggle/working/stage1_config.yaml").read())

weights: yolo11x_surv_v1.pt
conf_threshold: 0.2
imgsz: 640
names:
  0: Person
  1: Car
  2: Carrying_object
  3: Bike
  4: Gun
  5: Knife
nc: 6
val_map50: 0.6950273710349442
val_map50_95: 0.497443264293518



## 9. Export artifacts

Two files carry forward to Stage 1b. Everything else in `/kaggle/working/` is disposable.

In [14]:
shutil.copy2(BEST, "/kaggle/working/yolo11x_surv_v1.pt")

print("Artifacts for Stage 1b:")
for f in ["/kaggle/working/yolo11x_surv_v1.pt",
          "/kaggle/working/stage1_config.yaml"]:
    print(f"  {f}  ({os.path.getsize(f)/1e6:.1f} MB)")

print(f"\nTraining plots/curves: {RUNS}/{RUN_NAME}/")
print("\nSave a Kaggle Dataset from this notebook's output so Stage 1b can attach the weights.")

Artifacts for Stage 1b:
  /kaggle/working/yolo11x_surv_v1.pt  (114.4 MB)
  /kaggle/working/stage1_config.yaml  (0.0 MB)

Training plots/curves: /kaggle/working/runs/yolo11x_surv_v1/

Save a Kaggle Dataset from this notebook's output so Stage 1b can attach the weights.


---

## Next: Stage 1b — ByteTrack

Inputs it expects from this notebook:

- `yolo11x_surv_v1.pt`
- `stage1_config.yaml` (carries `conf_threshold` and the class map)

Output contract for Stage 1b:

```
video_id, camera_id, frame_id, timestamp_sec,
x1, y1, x2, y2, confidence, class_id, class_name, track_id
```